# Task 3: Prediction of perturbation effect

In [ ]:
import torch
import torch.nn as nn
import pytorch_lightning as pl
from pathlib import Path
import numpy as np
import pandas as pd

import scanpy as sc
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
# DataModule/DataSet/Baseline/condMLP/pertMLP/cVAE/MLP/train live here
from scmlcourse.perturbation import *

In [ ]:
data_dir = Path("../../../data")
pl.seed_everything(0)

### Data Loading


In [ ]:
# build the perturbation-prediction datamodule: top perturbed genes +
# 2000 HVGs, per-gene logFC targets, held-out test perturbations
data_module = DataModule(data_dir/"qced_data.h5ad", n_workers=8, n_top_genes=2000, retain_adata = True)

In [ ]:
print(
    data_module.adata,
    data_module.n_vars,
    data_module.batch_size,
    sep="\n------\n")

## Simplistic Baseline: Small MLP

In [ ]:
# sanity-check the train/test split: which (gene, condition) combinations
# ended up in the held-out test set vs. the training set
data_module.adata.obs["train_set"] = data_module.adata.obs["perturbed_condition"].map(lambda x: x in data_module.train_genes)
data_module.adata.obs[["perturbed_gene", "perturbation_2", "train_set"]].value_counts().reset_index().pivot(index="perturbed_gene", columns=["perturbation_2", "train_set"])

In [ ]:
from pytorch_lightning.loggers import CSVLogger
from pytorch_lightning.callbacks import ModelCheckpoint
log_dir = Path("../logs")
log_dir.mkdir(exist_ok=True)
checkpointer = ModelCheckpoint(
    dirpath=log_dir,
    filename="epoch:{epoch}-val_loss:{val_loss:.2f}",
    monitor=None,
    verbose=False
)

In [ ]:
# three baselines sharing the same MLP shape, differing in their input:
# - baseline_model: raw expression only
# - cond_model: one-hot perturbed-gene condition only
# - pert_model: expression + one-hot condition concatenated
mlp_config = dict(
    in_dim=data_module.n_vars,
    out_dim=data_module.n_vars,
    hidden_dims=(data_module.n_vars,)*3,
)
baseline_model = Baseline(mlp_config)
cond_model = condMLP(mlp_config)
pert_config = mlp_config.copy()
pert_config.update(in_dim=data_module.n_vars*2)
pert_model = pertMLP(pert_config)

In [ ]:
def get_last_version(dir:Path):
    """Return the most recent `version_*` subdirectory of `dir` (by trailing digit)."""
    dirs = [d for d in dir.glob("version_*")]
    v = [d.stem[-1]]
    return dirs[np.argmax(v)]
def get_ckpt(dir:Path, last = False):
    """Return a checkpoint path in `dir`: `last.ckpt` if `last` and present, else the lowest-val_loss checkpoint."""
    if last and (dir/"last.ckpt").exists():
        return dir/"last.ckpt"
    ckpts = [c for c in dir.glob("*.ckpt") if not "last" in c.name]
    vloss = [c.stem.split("=")[-1] for c in ckpts]
    return ckpts[np.argmin(vloss)]

# train all three baselines on the same data_module; each call trains,
# early-stops on val_loss, and tests on the best checkpoint
baseline_model, baseline_test_scores, baseline_logged_dir = train(
    baseline_model,
    data_module,
    max_epochs = 20,
    patience = 100,
    log_dir = "logs",
    run_name = "baseline",
    test_on="best",
    seed = 0,)
cond_model, cond_test_scores, cond_logged_dir = train(
    cond_model,
    data_module,
    max_epochs = 20,
    patience = 100,
    log_dir = "logs",
    run_name = "cond_baseline",
    test_on="best",
    seed = 0,)
pert_model, pert_test_scores, pert_logged_dir = train(
    pert_model,
    data_module,
    max_epochs = 20,
    patience = 15,
    log_dir = "logs",
    run_name = "pert_baseline",
    test_on="best",
    seed = 0,)

import gc
gc.collect()

In [ ]:
plot_cols = ["train_loss_epoch", "val_loss", "val_pearson"]

In [ ]:
# training curves for the expression-only baseline
logs = pd.read_csv(baseline_logged_dir+"/metrics.csv")
logs.groupby("epoch").agg(lambda x: x[~x.isna()].iloc[0] if (~x.isna()).sum()>0 else pd.NA).reset_index()[plot_cols+["epoch"]].dropna(axis=0, how="any").plot("epoch",plot_cols, sharey=False, subplots=True, figsize=(10,10))

In [ ]:
# training curves for the condition-only baseline
logs = pd.read_csv(cond_logged_dir+"/metrics.csv")
logs.groupby("epoch").agg(lambda x: x[~x.isna()].iloc[0] if (~x.isna()).sum()>0 else pd.NA).reset_index()[plot_cols+["epoch"]].dropna(axis=0, how="any").plot("epoch",plot_cols, sharey=False, subplots=True, figsize=(10,10))

In [ ]:
# training curves for the expression+condition baseline
logs = pd.read_csv(pert_logged_dir+"/metrics.csv")
logs.groupby("epoch").agg(lambda x: x[~x.isna()].iloc[0] if (~x.isna()).sum()>0 else pd.NA).reset_index()[plot_cols+["epoch"]].dropna(axis=0, how="any").plot("epoch",plot_cols, sharey=False, subplots=True, figsize=(10,10))

In [ ]:
# conditional VAE: encoder shrinks expression down to a 512-d latent,
# decoder expands [latent, perturbation_one_hot] back to expression
in_dim=data_module.n_vars
latent_dim = 512
dropout=0.1
encoder_kwargs = dict(
    hidden_dims=(in_dim, in_dim, in_dim, in_dim, in_dim, latent_dim, latent_dim),
    dropout=0
)
decoder_kwargs = dict(
    hidden_dims=(latent_dim+in_dim, latent_dim+in_dim, in_dim, in_dim, in_dim, in_dim),
    dropout=dropout
)
cvae = cVAE(in_dim = in_dim, latent_dim=latent_dim, encoder_kwargs = encoder_kwargs, decoder_kwargs = decoder_kwargs, lr=0.00001, kl_midpoint=10)


In [ ]:
# train the cVAE; test on the last (not best) checkpoint since it's
# evaluated via reconstruction+KL loss rather than logFC loss during training
model, test_scores, logged_dir = train(
    cvae,
    data_module,
    max_epochs = 20,
    patience = 100,
    log_dir = log_dir,
    run_name = "cvae",
    seed = 0,
    test_on="last"
)
import gc
gc.collect()

In [ ]:
logs = pd.read_csv(logged_dir+"/metrics.csv")
logs

In [ ]:
# cVAE training curves: reconstruction, KL, and total loss/pearson, plus
# the annealed KL weight and learning rate
plot_cols = ["train_loss_epoch", "val_loss", "train_loss_mse_x_epoch", "val_loss_mse_x", "train_loss_kl_epoch", "val_loss_kl", "train_pearson", "val_pearson", "kl_factor", "lr"]
logs.groupby("epoch").agg(lambda x: x[~x.isna()].iloc[0] if (~x.isna()).sum()>0 else pd.NA).reset_index()[plot_cols+["epoch"]].dropna(axis=0, how="any").plot("epoch",plot_cols, sharey=False, subplots=True, layout=(-1,2), figsize=(10,10))

In [ ]:
# diagnostic ranges for the cVAE's latent space and decoder output, to
# check for posterior collapse / saturation during training
plot_cols = ["train_mu_min",
"train_mu_max",
"train_mu_mean",
"train_logvar_min",
"train_logvar_max",
"train_logvar_mean",
"train_latent_min",
"train_latent_max",
"train_latent_mean",
"train_x_hat_min",
"train_x_hat_max",
"train_x_hat_mean",]
logs.groupby("epoch").agg(lambda x: x[~x.isna()].iloc[0] if (~x.isna()).sum()>0 else pd.NA).reset_index()[plot_cols+["epoch"]].dropna(axis=0, how="any").plot("epoch",plot_cols, sharey=False, subplots=True, layout=(4,3), figsize=(10,10))

In [ ]:
# collect held-out test metrics across all four models for comparison
metrics = {model:d[0] for model, d in zip(["Expression","Condition","Condition+Expression", "cVAE"], [baseline_test_scores, cond_test_scores, pert_test_scores, test_scores])}
metrics

In [ ]:
# bar plots comparing test loss and test Pearson correlation across models
# (the cVAE logs these under "pearson_fc"/"test_loss" via predict_fc, hence
# the fallback lookup by metric suffix)
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2)
for ax, m in zip(axes, ["test_loss","test_pearson"]):
    ax.bar(metrics.keys(), [v.get(m, v.get(m.split("_")[-1]+"_fc", None)) for v in metrics.values()])
    ax.set_ylabel(" ".join(m.split("_")).capitalize())
